Part 1: Load IMDb and TMDb datasets

In [98]:
import pandas as pd
import os

### Part 1: Load IMDb and TMDb datasets  

#### Part 1.1: Load cleaned IMDb movies  

In this part, we load the cleaned IMDb movies table that we exported from our first data preprocessing notebook. This table already keeps only titles with `titleType = "movie"` and contains the core attributes we will use as our main movies table in the database (for example: `movie_id`, `title`, `release_year`, `runtime_minutes`, `rating`, `num_votes`, and so on).  

We will use this table as the reference when aligning and filtering the TMDb dataset.


In [76]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [77]:
# Path to all files downloaded from imdb website
CLEANED_DIR = f"/content/drive/Shareddrives/CIS5500/Cleaned_Dataset"
BASE_DIR = f"/content/drive/Shareddrives/CIS5500/TMDB_Dataset"
MOVIES_PATH  = f"{CLEANED_DIR}/movies.csv"
TMDB_PATH = f"{BASE_DIR}/tmdb.csv"

In [78]:
movies = pd.read_csv(MOVIES_PATH)

#### Part 1.2: Load TMDb dataset (only the columns we need)  

The original TMDb file is large and contains many columns that we will not use in our analysis or database schema. To keep the processing efficient and focused, we load only the TMDb columns that are relevant for our project:

- `id`: TMDb’s own movie identifier (we will rename this to `tmdb_id`)
- `imdb_id`: IMDb ID, used to link TMDb entries to our IMDb `movies` table
- `budget` and `revenue`: financial information (used for ROI and revenue analysis)
- `overview`: plot summary text (useful for the frontend and movie details)
- `popularity`: TMDb popularity score
- `poster_path`: relative path to the poster image (useful for the frontend UI, combined with `https://image.tmdb.org/t/p/w500`)
- `production_countries`: list of production countries, stored as a raw JSON-like string for now  

We load only these columns at read time, using the `usecols` option, and then we will standardize their names.


In [79]:
tmdb_usecols = [
    "id",
    "imdb_id",
    "budget",
    "revenue",
    "overview",
    "popularity",
    "poster_path",
    "production_countries"
]

tmdb = pd.read_csv(TMDB_PATH, usecols=tmdb_usecols)

### Part 2: Initial TMDb cleaning and alignment fields  

#### Part 2.1: Standardize TMDb column names  

In this part, we create a working TMDb dataframe and standardize some column names to match the naming style we plan to use in our database schema. Specifically, we rename:

- `id` to `tmdb_id`  
- `overview` to `overview_tmdb`  
- `production_countries` to `production_countries_raw`  

This makes it clear that these columns are coming from the TMDb dataset and prepares them for further cleaning and integration with our IMDb `movies` table.


In [80]:
tmdb_work = tmdb.copy()

# Rename TMDb columns to more descriptive / schema-friendly names
tmdb_work = tmdb_work.rename(columns={
    "id": "tmdb_id",
    "production_countries": "production_countries_raw"
})

tmdb_work.head()

,tmdb_id,revenue,budget,imdb_id,overview,popularity,poster_path,production_countries_raw
0,27205,825532764,160000000,tt1375666,"Cobb, a skilled thief who commits corporate es...",83.95,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,"United Kingdom, United States of America"
1,157336,701729206,165000000,tt0816692,The adventures of a group of explorers who mak...,140.24,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,"United Kingdom, United States of America"
2,155,1004558444,185000000,tt0468569,Batman raises the stakes in his war on crime. ...,130.64,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,"United Kingdom, United States of America"
3,19995,2923706026,237000000,tt0499549,"In the 22nd century, a paraplegic Marine is di...",79.93,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,"United States of America, United Kingdom"
4,24428,1518815515,220000000,tt0848228,When an unexpected enemy emerges and threatens...,98.08,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,United States of America


#### Part 2.2: Drop rows without `imdb_id` and normalize the `imdb_id` format  

To join TMDb with our IMDb `movies` table, we need a valid `imdb_id` for each TMDb row. In this part, we:

- Remove TMDb rows that have a missing `imdb_id`  
- Clean the `imdb_id` field by converting it to a string type and stripping any leading or trailing whitespace  

This ensures that the join key is well-formed, consistent, and ready for the later entity resolution step.


In [81]:
before = tmdb_work.shape[0]

# Drop rows that have no imdb_id (we cannot link them to IMDb movies)
tmdb_work = tmdb_work[tmdb_work["imdb_id"].notna()].copy()

after = tmdb_work.shape[0]
print(f"Dropped {before - after} rows with missing imdb_id. Remaining: {after}")

# Normalize imdb_id to be a clean string (strip whitespace, consistent type)
tmdb_work["imdb_id"] = (
    tmdb_work["imdb_id"]
    .astype(str)
    .str.strip()
)

tmdb_work["imdb_id"].head()


Dropped 686027 rows with missing imdb_id. Remaining: 649035


,imdb_id
0,tt1375666
1,tt0816692
2,tt0468569
3,tt0499549
4,tt0848228


### Part 3: Align TMDb entries to our IMDb `movies` table (entity resolution)  

Our `movies` table from IMDb already contains only titles with `titleType = "movie"`. To make sure the TMDb data is aligned with this set, we keep only TMDb rows whose `imdb_id` appears in the IMDb `movies` dataframe.  

This step is the core of our entity resolution between the two datasets. After this filter, every TMDb row we keep corresponds to a movie that exists in our main IMDb-based schema.


In [82]:
# Collect the set of valid IMDb movie IDs from our cleaned movies table
valid_imdb_ids = set(movies["movie_id"])

before = tmdb_work.shape[0]

# Keep only TMDb rows that correspond to movies we actually have in our IMDb dataset
tmdb_work = tmdb_work[tmdb_work["imdb_id"].isin(valid_imdb_ids)].copy()

after = tmdb_work.shape[0]

print(f"After matching to IMDb movies: {after} rows (dropped {before - after})")
print("Unique imdb_id count:", tmdb_work["imdb_id"].nunique())


After matching to IMDb movies: 330500 rows (dropped 318535)
Unique imdb_id count: 329525


### Part 4: Handle duplicate `imdb_id` entries in TMDb  

Even after aligning TMDb to our IMDb `movies` table, there can still be cases where the same `imdb_id` appears multiple times in the TMDb dataset (for example, different cuts, re-releases, or data quirks).  

For our integrated schema, we want at most one TMDb record per IMDb movie, while still keeping `tmdb_id` as the primary key on the TMDb side. In this part, we:

- Check how many `imdb_id` values appear more than once in `tmdb_work`
- When duplicates exist, sort TMDb rows so that the most informative record comes first (higher `popularity` and higher `revenue`)
- Drop duplicates based on `imdb_id`, keeping only the first row for each `imdb_id`  

After this step, each IMDb movie will map to at most one TMDb row, but `tmdb_id` remains the key for the TMDb table.


In [83]:
# Check how many imdb_id values appear more than once
dup_counts = tmdb_work["imdb_id"].value_counts()
num_multi = (dup_counts > 1).sum()
print(f"Number of imdb_id values that appear more than once: {num_multi}")

# If duplicates exist, keep the "best" row per imdb_id:
# sort by imdb_id, then popularity (descending), then revenue (descending)
tmdb_work = (
    tmdb_work
    .sort_values(["imdb_id", "popularity", "revenue"], ascending=[True, False, False])
    .drop_duplicates(subset=["imdb_id"], keep="first")
)

print("After deduplication:")
print("Rows:", tmdb_work.shape[0])
print("Unique imdb_id:", tmdb_work["imdb_id"].nunique())


Number of imdb_id values that appear more than once: 475
After deduplication:
Rows: 329525
Unique imdb_id: 329525


### Part 5: Clean TMDb numeric and text fields  

#### Part 5.1: Convert numeric columns and format with two decimal places  

In the TMDb raw data, the numeric fields are read as strings, and many movies have `budget = 0` or `revenue = 0`. In this dataset, these zeros usually mean "unknown" rather than a true value of 0.  

In this part, we:

- Convert `budget`, `revenue`, and `popularity` to numeric float types.
- For `budget` and `revenue`, interpret 0 as missing and replace it with `NaN`.
- Keep `popularity` as a float (no special handling for 0).
- Set a global display option so that all floats are shown with exactly 2 digits after the decimal point (and no scientific notation) in the notebook.  

This keeps the financial and popularity fields consistent, semantically correct, and easy to read.


In [84]:
# Convert all three numeric columns to float
tmdb_work["popularity"] = pd.to_numeric(tmdb_work["popularity"], errors="coerce")
tmdb_work["budget"] = pd.to_numeric(tmdb_work["budget"], errors="coerce")
tmdb_work["revenue"] = pd.to_numeric(tmdb_work["revenue"], errors="coerce")

# For budget and revenue, treat 0 as missing (unknown)
for col in ["budget", "revenue"]:
    tmdb_work.loc[tmdb_work[col] == 0, col] = pd.NA
    # After replacing with NaN/NA, keep as float (pandas will use float dtype)
    tmdb_work[col] = tmdb_work[col].astype("float64")

# Optional: round values to 2 decimal places (mainly relevant if any non-integer)
tmdb_work["budget"] = tmdb_work["budget"].round(2)
tmdb_work["revenue"] = tmdb_work["revenue"].round(2)
tmdb_work["popularity"] = tmdb_work["popularity"].round(2)

# Set global float display: always show 2 digits after decimal, no scientific notation
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

print("Summary of numeric columns:")
print(tmdb_work[["budget", "revenue", "popularity"]].describe())


Summary of numeric columns:
            budget       revenue  popularity
count     26258.00      15977.00   329525.00
mean   10609114.95   44572925.67        2.40
std    27964764.44  128210151.54       12.82
min           1.00          1.00        0.00
25%       43263.25     473104.00        0.60
50%     1000000.00    4840034.00        0.88
75%     8000000.00   29433930.00        1.75
max   888000000.00 2923706026.00     2994.36


### Part 6: Column-level sanity checks for the cleaned TMDb data  

Before finalizing the `tmdb_movies` table and exporting it to CSV, we perform a set of sanity checks on each column in `tmdb_work`. The goal is to confirm that:

- Data types are as expected (numeric vs text).
- There are no unexpected missing values in key columns (such as `tmdb_id` or `imdb_id`).
- The percentage of missing values in `budget` and `revenue` matches our expectations after treating 0 as unknown.
- There are no duplicate `tmdb_id` values.
- Numeric fields (`budget`, `revenue`, `popularity`) do not contain clearly invalid values (for example, negative budgets).  

These checks help us catch potential issues early and give us a clean story to describe in the data cleaning section of the report.


In [85]:
# Basic info: dtypes and non-null counts
print("=== tmdb_work.info() ===")
tmdb_work.info()
print("\n")

# Build a small summary table for each column
summary = pd.DataFrame({
    "dtype": tmdb_work.dtypes,
    "n_non_null": tmdb_work.notna().sum(),
    "n_null": tmdb_work.isna().sum(),
    "pct_null": (tmdb_work.isna().mean() * 100).round(2),
    "n_unique": tmdb_work.nunique()
})

print("=== Column-wise summary ===")
summary


=== tmdb_work.info() ===
<class 'pandas.core.frame.DataFrame'>
Index: 329525 entries, 175059 to 460987
Data columns (total 8 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   tmdb_id                   329525 non-null  int64  
 1   revenue                   15977 non-null   float64
 2   budget                    26258 non-null   float64
 3   imdb_id                   329525 non-null  object 
 4   overview                  290002 non-null  object 
 5   popularity                329525 non-null  float64
 6   poster_path               270793 non-null  object 
 7   production_countries_raw  243401 non-null  object 
dtypes: float64(3), int64(1), object(4)
memory usage: 22.6+ MB


=== Column-wise summary ===


,dtype,n_non_null,n_null,pct_null,n_unique
tmdb_id,int64,329525,0,0.00,329525
revenue,float64,15977,313548,95.15,13428
budget,float64,26258,303267,92.03,3697
imdb_id,object,329525,0,0.00,329525
overview,object,290002,39523,11.99,287201
popularity,float64,329525,0,0.00,4481
poster_path,object,270793,58732,17.82,270666
production_countries_raw,object,243401,86124,26.14,7954


In [86]:
print("=== Identifier checks ===")

# tmdb_id should be non-null and unique
dup_tmdb_id = tmdb_work["tmdb_id"].duplicated().sum()
null_tmdb_id = tmdb_work["tmdb_id"].isna().sum()

print(f"Duplicate tmdb_id count: {dup_tmdb_id}")
print(f"Null tmdb_id count: {null_tmdb_id}")

# imdb_id should be non-null (we dropped missing ones earlier)
null_imdb_id = tmdb_work["imdb_id"].isna().sum()
print(f"Null imdb_id count (after cleaning): {null_imdb_id}")

# Quick look at a few rows
tmdb_work[["tmdb_id", "imdb_id"]].head()

=== Identifier checks ===
Duplicate tmdb_id count: 0
Null tmdb_id count: 0
Null imdb_id count (after cleaning): 0


,tmdb_id,imdb_id
175059,356151,tt0000009
50197,147610,tt0000147
335899,415440,tt0000335
39199,20105,tt0000574
986470,396922,tt0000591


In [87]:
print("=== Numeric sanity checks ===")

for col in ["budget", "revenue", "popularity"]:
    print(f"\nColumn: {col}")
    print("  Min:", tmdb_work[col].min())
    print("  Max:", tmdb_work[col].max())
    print("  Null count:", tmdb_work[col].isna().sum())

# Show a few rows where budget and revenue are non-null to confirm they look reasonable
tmdb_work[tmdb_work["budget"].notna()][["tmdb_id", "imdb_id", "budget", "revenue", "popularity"]].head(10)


=== Numeric sanity checks ===

Column: budget
  Min: 1.0
  Max: 888000000.0
  Null count: 303267

Column: revenue
  Min: 1.0
  Max: 2923706026.0
  Null count: 313548

Column: popularity
  Min: 0.0
  Max: 2994.36
  Null count: 0


,tmdb_id,imdb_id,budget,revenue,popularity
1106244,988601,tt0001530,987.00,NaN,0.60
1139083,943338,tt0001604,800.00,NaN,0.65
619275,1459889,tt0001911,2577.00,NaN,0.01
96584,46758,tt0002461,30000.00,NaN,1.71
316978,277699,tt0003301,50000.00,NaN,0.60
54658,96128,tt0003471,5700.00,1800000.00,1.25
49427,62935,tt0004099,23.00,NaN,2.73
846096,572090,tt0004391,2221000.00,5773000.00,0.90
1076212,468031,tt0004545,16988.00,87028.00,0.90
6584,618,tt0004972,100000.00,11000000.00,13.79


In [89]:
print("=== Sample text values ===")

print("\nSample overview values:")
print(tmdb_work["overview"].dropna().head(3))

print("\nSample poster_path values:")
print(tmdb_work["poster_path"].dropna().head(3))

print("\nSample production_countries_raw values:")
print(tmdb_work["production_countries_raw"].dropna().head(3))


=== Sample text values ===

Sample overview values:
175059    The adventures of a female reporter in the 1890s.
50197     This legendary fight was filmed on March 17, 1...
335899    The plot outlines the story of the early Chris...
Name: overview, dtype: object

Sample poster_path values:
175059    /eDRXkbGYQf4TIrha6Akg24mZuXN.jpg
50197     /nL7aYlGYJZU7Gx8kFu9bTp3QYRu.jpg
39199     /4CHQV5nRoHwpaS8LaGNxTh8V0ad.jpg
Name: poster_path, dtype: object

Sample production_countries_raw values:
175059    United States of America
50197     United States of America
335899                   Australia
Name: production_countries_raw, dtype: object


### Part 7: Normalize production countries into separate tables  

The `production_countries_raw` column in `tmdb_work` contains a comma-separated list of country names for each movie, for example:  

- `"United Kingdom, United States of America"`  

Storing this list as a single text field violates normalization, because multiple values are embedded in one attribute. In this part, we normalize production countries into two separate tables:

1. `countries(country_name)`  
   - One row per distinct country name appearing in the TMDb dataset.  
   - `country_name` serves as the primary key.  

2. `movie_production_countries(tmdb_id, country_name)`  
   - One row per (movie, country) pair.  
   - `tmdb_id` is a foreign key referencing `tmdb_movies(tmdb_id)`.  
   - `country_name` is a foreign key referencing `countries(country_name)`.  
   - The composite key (`tmdb_id`, `country_name`) ensures there are no duplicate country assignments for a movie.  

This design removes the multi-valued attribute from `tmdb_movies`, puts country names in their own table, and creates a clean many-to-many relationship between movies and production countries.


Part 7.1 – Explode `production_countries_raw` into long format

In [90]:
# Start from tmdb_id and production_countries_raw
countries_long = tmdb_work[["tmdb_id", "production_countries_raw"]].copy()

# Keep only rows where production_countries_raw is not null
countries_long = countries_long[countries_long["production_countries_raw"].notna()].copy()

# Split the comma-separated country names into Python lists
countries_long["country_list"] = countries_long["production_countries_raw"].astype(str).str.split(",")

# Explode the list so each (tmdb_id, country_name) becomes its own row
countries_long = countries_long.explode("country_list")

# Clean up the country names: strip whitespace
countries_long["country_name"] = countries_long["country_list"].astype(str).str.strip()

# Drop rows where country_name ended up empty
countries_long = countries_long[countries_long["country_name"] != ""].copy()

# Drop helper column we no longer need
countries_long = countries_long.drop(columns=["country_list"])

# Remove duplicate (tmdb_id, country_name) pairs if any
countries_long = countries_long.drop_duplicates(subset=["tmdb_id", "country_name"])

print("countries_long shape (tmdb_id, country_name pairs):", countries_long.shape)
countries_long.head(10)


countries_long shape (tmdb_id, country_name pairs): (284729, 3)


,tmdb_id,production_countries_raw,country_name
175059,356151,United States of America,United States of America
50197,147610,United States of America,United States of America
335899,415440,Australia,Australia
39199,20105,Australia,Australia
986470,396922,France,France
709279,1367677,Italy,Italy
242299,470664,United States of America,United States of America
730105,1371775,Spain,Spain
335453,415470,United Kingdom,United Kingdom
798122,1439199,Denmark,Denmark


Part 7.2 – Build countries table

In [91]:
# One row per distinct country name
countries_df = (
    countries_long[["country_name"]]
    .drop_duplicates()
    .sort_values("country_name")
    .reset_index(drop=True)
)

print("countries_df shape (distinct countries):", countries_df.shape)
countries_df.head(10)


countries_df shape (distinct countries): (235, 1)


,country_name
0,Afghanistan
1,Albania
2,Algeria
3,American Samoa
4,Andorra
5,Angola
6,Anguilla
7,Antarctica
8,Antigua and Barbuda
9,Argentina


Part 7.3 – Build movie_production_countries table

In [94]:
movie_production_countries_df = countries_long[["tmdb_id", "country_name"]].copy()

print("movie_production_countries_df shape:", movie_production_countries_df.shape)
movie_production_countries_df.head(10)

movie_production_countries_df shape: (284729, 2)


,tmdb_id,country_name
175059,356151,United States of America
50197,147610,United States of America
335899,415440,Australia
39199,20105,Australia
986470,396922,France
709279,1367677,Italy
242299,470664,United States of America
730105,1371775,Spain
335453,415470,United Kingdom
798122,1439199,Denmark


#### Part 7.4: Sanity checks for `countries` and `movie_production_countries`  

Before exporting the normalized country tables, we perform a few sanity checks:

- Confirm that `countries_df` has only distinct `country_name` values and no unexpected nulls or empty strings.
- Check how many TMDb movies have at least one production country versus none.
- Confirm there are no duplicate (tmdb_id, country_name) pairs in `movie_production_countries_df`.
- Look at the most frequent countries to make sure the values look reasonable (for example, "United States of America", "United Kingdom", etc.).  

These checks help validate that the normalization of production countries worked as intended.


In [93]:
print("=== countries_df checks ===")
print("Shape (distinct countries):", countries_df.shape)

# Check for nulls or empty strings in country_name
null_countries = countries_df["country_name"].isna().sum()
empty_countries = (countries_df["country_name"].astype(str).str.strip() == "").sum()
print("Null country_name count:", null_countries)
print("Empty country_name count:", empty_countries)

print("\nSample countries:")
print(countries_df.head(10))


print("\n=== movie_production_countries_df checks ===")
print("Shape (tmdb_id, country_name pairs):", movie_production_countries_df.shape)

# Check for duplicate (tmdb_id, country_name) pairs
dup_pairs = movie_production_countries_df.duplicated(subset=["tmdb_id", "country_name"]).sum()
print("Duplicate (tmdb_id, country_name) pairs:", dup_pairs)

# How many tmdb_id have at least one country?
movies_with_country = movie_production_countries_df["tmdb_id"].nunique()
total_tmdb_movies = tmdb_work["tmdb_id"].nunique()
print(f"TMDb movies with at least one production country: {movies_with_country} / {total_tmdb_movies}")

# Top 10 most frequent countries
print("\nTop 10 most frequent production countries:")
print(
    movie_production_countries_df["country_name"]
    .value_counts()
    .head(10)
)


=== countries_df checks ===
Shape (distinct countries): (235, 1)
Null country_name count: 0
Empty country_name count: 0

Sample countries:
          country_name
0          Afghanistan
1              Albania
2              Algeria
3       American Samoa
4              Andorra
5               Angola
6             Anguilla
7           Antarctica
8  Antigua and Barbuda
9            Argentina

=== movie_production_countries_df checks ===
Shape (tmdb_id, country_name pairs): (284729, 2)
Duplicate (tmdb_id, country_name) pairs: 0
TMDb movies with at least one production country: 243401 / 329525

Top 10 most frequent production countries:
country_name
United States of America    70680
France                      17201
India                       15266
Germany                     14559
United Kingdom              14534
Japan                       13152
Italy                       11496
Spain                        7944
Canada                       7359
Mexico                       7133
Name: c

### Part 8: Build `MarketInfo` table from cleaned TMDb data  

#### Part 8.1: Select columns for `MarketInfo`  

We now construct the dataframe that will become our `MarketInfo` table in PostgreSQL. This table contains commercial and market-facing information about each movie, sourced from TMDb, and linked back to our main `Movies` table via `imdb_id`.  

From `tmdb_work`, we select the following columns:

- `tmdb_id`: TMDb movie identifier (will be the primary key of `MarketInfo`)
- `imdb_id`: IMDb movie identifier (foreign key to `Movies.movie_id`)
- `budget`: production budget (numeric, with 0 already treated as missing)
- `revenue`: box office revenue (numeric, with 0 already treated as missing)
- `popularity`: TMDb popularity score
- `overview`: plot summary text from TMDb
- `poster_path`: relative path to the movie poster on TMDb  

We store this subset in a new dataframe called `marketinfo`. This dataframe is what we will eventually export and load into the `MarketInfo` table in PostgreSQL.


In [95]:
# Select final columns for the MarketInfo table
marketinfo = tmdb_work[[
    "tmdb_id",
    "imdb_id",
    "budget",
    "revenue",
    "popularity",
    "overview",
    "poster_path"
]].copy()

print("marketinfo shape:", marketinfo.shape)
marketinfo.head()


marketinfo shape: (329525, 7)


,tmdb_id,imdb_id,budget,revenue,popularity,overview,poster_path
175059,356151,tt0000009,NaN,NaN,0.75,The adventures of a female reporter in the 1890s.,/eDRXkbGYQf4TIrha6Akg24mZuXN.jpg
50197,147610,tt0000147,NaN,NaN,1.25,"This legendary fight was filmed on March 17, 1...",/nL7aYlGYJZU7Gx8kFu9bTp3QYRu.jpg
335899,415440,tt0000335,NaN,NaN,0.60,The plot outlines the story of the early Chris...,NaN
39199,20105,tt0000574,NaN,NaN,2.16,Just as Galeen and Wegener's Der Golem (1915) ...,/4CHQV5nRoHwpaS8LaGNxTh8V0ad.jpg
986470,396922,tt0000591,NaN,NaN,0.60,The first feature-length motion picture produc...,NaN


#### Part 8.2: Sanity checks for `MarketInfo`  

Before exporting, we perform a few sanity checks on the `marketinfo` dataframe to confirm that:

- Data types match our expectations (identifiers as integer/string, financial fields as numeric, text fields as `object`/`string`).
- There are no nulls in `tmdb_id` or `imdb_id`, since these are key fields.
- `tmdb_id` is unique (so it can safely serve as the primary key of `MarketInfo`).
- The proportion of missing values in `budget` and `revenue` is reasonable given that 0 values were treated as unknown and converted to nulls.  

These checks help us verify that `MarketInfo` is ready to be exported and loaded into PostgreSQL.


In [96]:
print("=== marketinfo.info() ===")
marketinfo.info()
print("\n")

# Column-wise summary
mi_summary = pd.DataFrame({
    "dtype": marketinfo.dtypes,
    "n_non_null": marketinfo.notna().sum(),
    "n_null": marketinfo.isna().sum(),
    "pct_null": (marketinfo.isna().mean() * 100).round(2),
    "n_unique": marketinfo.nunique()
})

print("=== Column-wise summary for marketinfo ===")
mi_summary


=== marketinfo.info() ===
<class 'pandas.core.frame.DataFrame'>
Index: 329525 entries, 175059 to 460987
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   tmdb_id      329525 non-null  int64  
 1   imdb_id      329525 non-null  object 
 2   budget       26258 non-null   float64
 3   revenue      15977 non-null   float64
 4   popularity   329525 non-null  float64
 5   overview     290002 non-null  object 
 6   poster_path  270793 non-null  object 
dtypes: float64(3), int64(1), object(3)
memory usage: 20.1+ MB


=== Column-wise summary for marketinfo ===


,dtype,n_non_null,n_null,pct_null,n_unique
tmdb_id,int64,329525,0,0.00,329525
imdb_id,object,329525,0,0.00,329525
budget,float64,26258,303267,92.03,3697
revenue,float64,15977,313548,95.15,13428
popularity,float64,329525,0,0.00,4481
overview,object,290002,39523,11.99,287201
poster_path,object,270793,58732,17.82,270666


In [97]:
print("=== Key checks for MarketInfo ===")

# tmdb_id: should be non-null and unique
dup_tmdb_id = marketinfo["tmdb_id"].duplicated().sum()
null_tmdb_id = marketinfo["tmdb_id"].isna().sum()
print(f"Duplicate tmdb_id count: {dup_tmdb_id}")
print(f"Null tmdb_id count: {null_tmdb_id}")

# imdb_id: should be non-null (we filtered earlier on tmdb_work)
null_imdb_id = marketinfo["imdb_id"].isna().sum()
print(f"Null imdb_id count: {null_imdb_id}")

# Quick sample
marketinfo.head(10)


=== Key checks for MarketInfo ===
Duplicate tmdb_id count: 0
Null tmdb_id count: 0
Null imdb_id count: 0


,tmdb_id,imdb_id,budget,revenue,popularity,overview,poster_path
175059,356151,tt0000009,NaN,NaN,0.75,The adventures of a female reporter in the 1890s.,/eDRXkbGYQf4TIrha6Akg24mZuXN.jpg
50197,147610,tt0000147,NaN,NaN,1.25,"This legendary fight was filmed on March 17, 1...",/nL7aYlGYJZU7Gx8kFu9bTp3QYRu.jpg
335899,415440,tt0000335,NaN,NaN,0.60,The plot outlines the story of the early Chris...,NaN
39199,20105,tt0000574,NaN,NaN,2.16,Just as Galeen and Wegener's Der Golem (1915) ...,/4CHQV5nRoHwpaS8LaGNxTh8V0ad.jpg
986470,396922,tt0000591,NaN,NaN,0.60,The first feature-length motion picture produc...,NaN
292941,603786,tt0000615,NaN,NaN,0.60,Australian bushranger movie. The first filmed...,NaN
709279,1367677,tt0000630,NaN,NaN,0.60,An Italian short movie.,NaN
242299,470664,tt0000679,NaN,NaN,0.78,L. Frank Baum would appear in a white suit and...,/jecGhyYtPGrlVh32OFbDZF9KaTZ.jpg
730105,1371775,tt0000941,NaN,NaN,1.15,Felipe I el Hermoso provokes with his behavior...,/zJwGs5N5SBNJ1jGDMXdTHgeUkkh.jpg
335453,415470,tt0001028,NaN,NaN,0.60,A man is obsessed by the 'Salome' dance.,NaN


### Part 9: Export cleaned TMDb-based tables to CSV  

At this point, we have three cleaned dataframes derived from the TMDb dataset:

1. `marketinfo`  
   - Commercial and market-facing information for each movie (budget, revenue, popularity, overview, poster).  
   - Linked to our main `Movies` table via `imdb_id`.  
   - Will be loaded as the `MarketInfo` table in PostgreSQL.

2. `countries_df`  
   - One row per distinct production country (`country_name`).  
   - Will be loaded as the `Countries` table.

3. `movie_production_countries_df`  
   - One row per (tmdb_id, country_name) pair.  
   - Represents the many-to-many relationship between movies and production countries.  
   - Will be loaded as the `MovieProductionCountries` table.  

In this part, we export these three dataframes as CSV files into the same `Cleaned_Datasets` folder as our other cleaned IMDb-based tables.


In [100]:
MARKETINFO_PATH = os.path.join(CLEANED_DIR, "marketinfo.csv")
COUNTRIES_PATH = os.path.join(CLEANED_DIR, "countries.csv")
MOVIE_COUNTRIES_PATH = os.path.join(CLEANED_DIR, "movie_production_countries.csv")

marketinfo.to_csv(MARKETINFO_PATH, index=False)
countries_df.to_csv(COUNTRIES_PATH, index=False)
movie_production_countries_df.to_csv(MOVIE_COUNTRIES_PATH, index=False)

print("Saved marketinfo to:", MARKETINFO_PATH)
print("Saved countries to:", COUNTRIES_PATH)
print("Saved movie_production_countries to:", MOVIE_COUNTRIES_PATH)


Saved marketinfo to: /content/drive/Shareddrives/CIS5500/Cleaned_Dataset/marketinfo.csv
Saved countries to: /content/drive/Shareddrives/CIS5500/Cleaned_Dataset/countries.csv
Saved movie_production_countries to: /content/drive/Shareddrives/CIS5500/Cleaned_Dataset/movie_production_countries.csv
